In [2]:
# ========================
# PARTE 0: MONTAR DRIVE E INSTALAR DEPENDÊNCIAS
# ========================
from google.colab import drive
drive.mount('/content/drive')

!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium pandas openpyxl requests > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGS
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import os, time, requests, zipfile, re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--blink-settings=imagesEnabled=false")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 7)

# ========================
# PARTE 2: CONFIG ALGOLIA
# ========================
ALGOLIA_APP_ID = "1CF3ZT43ZU"
ALGOLIA_API_KEY = "150c68d1c61fc1835826a57a203dab72"
ALGOLIA_INDEX = "production_products"

headers_algolia = {
    "Content-Type": "application/json",
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key": ALGOLIA_API_KEY,
}

termos_busca = ["pisos", "revestimentos", "porcelanatos"]
modo = input("Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): ").strip()
modo = 1.0 if modo == "100" else 0.5

print("\n🔍 Coletando links...")
def buscar_links_por_categoria(termos):
    todos_links = []
    for termo in termos:
        links = []
        pagina = 0
        print(f"🔎 Buscando: {termo}")
        while True:
            url = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/{ALGOLIA_INDEX}/query"
            payload = { "params": f"query={termo}&page={pagina}&hitsPerPage=250" }
            res = requests.post(url, headers=headers_algolia, json=payload)
            if res.status_code != 200:
                break
            data = res.json()
            hits = data.get("hits", [])
            if not hits:
                break
            for item in hits:
                relative_url = item.get("url")
                if relative_url:
                    full_url = f"https://www.leroymerlin.com.br{relative_url}" if not relative_url.startswith("http") else relative_url
                    if full_url not in links:
                        links.append(full_url)
            pagina += 1
            if pagina >= data.get("nbPages", 0):
                break
        limite = int(len(links) * modo)
        todos_links.extend(links[:limite])
        print(f"🔗 {len(links[:limite])} links adicionados de {termo}")
    return todos_links

todos_links = buscar_links_por_categoria(termos_busca)
print(f"✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE PRODUTO
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}
    dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text if driver.find_elements(By.CSS_SELECTOR, "h1") else None
    dados['Código'] = url.split("_")[-1] if "_" in url else None

    preco_txt = None
    seletores = [
        '[data-cy="product-price"]', '[data-cy="product-price-integer"]',
        '[data-cy="product-price-sale"]', '.price__sales', '.value__integer',
        '.price', '.price__value', '.sales-price', '.main-price']
    for sel in seletores:
        elems = driver.find_elements(By.CSS_SELECTOR, sel)
        if elems:
            text = elems[0].text.strip()
            if text and text.lower() != "grátis":
                preco_txt = text
                break

    if not preco_txt:
        html = driver.page_source
        match = re.search(r"R\$ ?\d{1,3}(?:\.\d{3})*,\d{2}", html)
        if match:
            preco_txt = match.group()

    if preco_txt:
        try:
            preco_val = float(preco_txt.replace("R$", "").replace("/m²", "").replace(" ", "").replace(".", "").replace(",", "."))
            dados['Preço Promocional'] = f"{preco_val:.2f}".replace(".", ",")
        except:
            dados['Preço Promocional'] = preco_txt
    else:
        dados['Preço Promocional'] = "N/A"

    dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[name="description"]') else None
    dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content") if driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:image"]') else "N/A"

    specs = {}
    for tr in driver.find_elements(By.CSS_SELECTOR, 'table tr'):
        try:
            k = tr.find_element(By.CSS_SELECTOR, 'th').text.strip()
            v = tr.find_element(By.CSS_SELECTOR, 'td').text.strip()
            specs[k] = v
        except:
            continue
    dados['Especificações'] = "\n".join(f"{k}: {v}" for k, v in specs.items())
    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETA E SALVAMENTO A CADA 100
# ========================
data_nome = datetime.now().strftime("%Y-%m-%d")
base_drive = f"/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/{data_nome}"
pasta_imagens = os.path.join(base_drive, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

todos = []
for i, link in enumerate(todos_links, 1):
    try:
        dados = extrair_dados_produto(link)
        codigo = dados.get("Código", f"produto_{i}")
        caminho_img = os.path.join(pasta_imagens, f"{codigo}.png")
        if not os.path.exists(caminho_img) and dados.get("Imagem") not in [None, "N/A"]:
            img = requests.get(dados['Imagem'], stream=True, timeout=10)
            if img.status_code == 200:
                with open(caminho_img, 'wb') as f:
                    for chunk in img.iter_content(1024):
                        f.write(chunk)

        todos.append(dados)

        if i % 100 == 0 or i == len(todos_links):
            df = pd.DataFrame(todos)
            df.drop(columns=[c for c in df.columns if c in ['Erro']], inplace=True, errors='ignore')
            colunas = ['Código', 'Nome', 'Preço Promocional', 'Descrição', 'Especificações', 'Imagem', 'URL', 'Loja']
            df = df[colunas]
            df.to_csv(os.path.join(base_drive, f"leroymerlin_parcial_{i}.csv"), index=False, sep=";")
            df.to_excel(os.path.join(base_drive, f"leroymerlin_parcial_{i}.xlsx"), index=False)
            print(f"💾 Dados salvos com {i} produtos.")
            todos.clear()

    except Exception as e:
        print(f"❌ Erro no produto {i}:", e)

# ========================
# PARTE 5: UNIFICAR ARQUIVOS CSV E EXCEL
# ========================
print("\n📁 Unificando arquivos CSV...")
csvs = [f for f in os.listdir(base_drive) if f.endswith(".csv") and "FINAL" not in f]
df_final = pd.DataFrame()

for csv in csvs:
    df = pd.read_csv(os.path.join(base_drive, csv), sep=";")
    df_final = pd.concat([df_final, df], ignore_index=True)

csv_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.csv")
excel_final = os.path.join(base_drive, f"leroymerlin_FINAL_{data_nome}.xlsx")

df_final.to_csv(csv_final, index=False, sep=";")
df_final.to_excel(excel_final, index=False)

print(f"✅ CSV final salvo: {csv_final}")
print(f"✅ Excel final salvo: {excel_final}")


Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
Deseja coletar 100% ou 50% dos produtos? (digite 100 ou 50): 50

🔍 Coletando links...
🔎 Buscando: pisos
🔗 500 links adicionados de pisos
🔎 Buscando: revestimentos
🔗 500 links adicionados de revestimentos
🔎 Buscando: porcelanatos
🔗 500 links adicionados de porcelanatos
✅ Total de links coletados: 1500

❌ Erro no produto 12: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=138.0.7204.168); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
#0 0x5c0624e3287a <unknown>
#1 0x5c06248d72e0 <unknown>
#2 0x5c06248ddf28 <unknown>
#

❌ Erro no produto 58: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7ed8790>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 59: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b80979d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 60: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b86c3410>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 61: HTTPConnectionPool(host='localhost',

❌ Erro no produto 76: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7edaed0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 77: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7b20c10>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 78: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7b51890>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 79: HTTPConnectionPool(host='localhost',

❌ Erro no produto 95: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b86c2090>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 96: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b86c1b10>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 97: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7a972d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 98: HTTPConnectionPool(host='localhost',

❌ Erro no produto 106: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b9997010>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 107: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b8097c10>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 108: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b9996fd0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 109: HTTPConnectionPool(host='localho

❌ Erro no produto 122: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7ab4d50>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 123: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7a959d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 124: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b9995b90>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 125: HTTPConnectionPool(host='localho

... (41 outputs left out)

❌ Erro no produto 1385: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6ba22cf50>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1386: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7ab6b50>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1387: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b8009310>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1388: HTTPConnectionPool(host='loc

❌ Erro no produto 1414: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b9995e50>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1415: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7fe7010>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1416: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7f381d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1417: HTTPConnectionPool(host='loc

❌ Erro no produto 1452: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7f3add0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1453: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7fd1dd0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1454: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7f53450>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1455: HTTPConnectionPool(host='loc

❌ Erro no produto 1485: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b79703d0>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1486: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7f33990>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1487: HTTPConnectionPool(host='localhost', port=33399): Max retries exceeded with url: /session/58f9595738ad1ec8c5d27ddaed960866/url (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb6b7f99c50>: Failed to establish a new connection: [Errno 111] Connection refused'))
❌ Erro no produto 1488: HTTPConnectionPool(host='loc

In [ ]:
# ========================
# UNIFICAR
# ========================

from google.colab import drive
drive.mount('/content/drive')

# ========================
# UNIR CSVs DO DRIVE + UPLOAD MANUAL E GERAR EXCEL
# ========================

import pandas as pd
import os
from datetime import datetime
from google.colab import files

# Caminho base do Drive
base_drive = "/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY"
pastas_data = [os.path.join(base_drive, p) for p in os.listdir(base_drive) if os.path.isdir(os.path.join(base_drive, p))]

# Lista para armazenar todos os DataFrames
todos_dfs = []

print("📂 Lendo arquivos do Google Drive...\n")

# 1. Coletar CSVs do Google Drive
for pasta in pastas_data:
    for arquivo in os.listdir(pasta):
        if arquivo.endswith(".csv") and "leroymerelin" in arquivo:
            caminho_csv = os.path.join(pasta, arquivo)
            try:
                df = pd.read_csv(caminho_csv, sep=";")
                todos_dfs.append(df)
                print(f"✅ Adicionado do Drive: {caminho_csv}")
            except Exception as e:
                print(f"⚠️ Erro ao ler {caminho_csv}: {e}")

# 2. Upload manual de arquivos
print("\n⬆️ Faça upload de arquivos CSV adicionais (opcional)")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".csv"):
        try:
            df_upload = pd.read_csv(filename, sep=";")
            todos_dfs.append(df_upload)
            print(f"✅ Adicionado do upload: {filename}")
        except Exception as e:
            print(f"⚠️ Erro ao ler {filename}: {e}")

# 3. Verificação final
if not todos_dfs:
    print("❌ Nenhum arquivo CSV foi encontrado ou carregado.")
else:
    # 4. Concatenar tudo
    df_final = pd.concat(todos_dfs, ignore_index=True)

    # 5. Remover duplicatas pela coluna Código, se existir
    if 'Código' in df_final.columns:
        df_final.drop_duplicates(subset='Código', inplace=True)

    # 6. Ordenar se possível
    if 'Nome' in df_final.columns:
        df_final.sort_values(by="Nome", inplace=True)

    # 7. Salvar como Excel
    data_hoje = datetime.now().strftime("%Y-%m-%d")
    caminho_excel = os.path.join(base_drive, f"LERoyMERLIN_UNIFICADO_UPLOAD_{data_hoje}.xlsx")
    df_final.to_excel(caminho_excel, index=False)

    print(f"\n✅ Excel final salvo em:\n{caminho_excel}")


Mounted at /content/drive
📂 Lendo arquivos do Google Drive...

✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_100.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_200.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_300.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_400.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_500.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/2025-07-12/leroymerelin_2025-07-12_600.csv
✅ Adicionado do Drive: /content/drive/MyDrive/WebScraping/F


✅ Excel final salvo em:
/content/drive/MyDrive/WebScraping/FEITO/Leroy/COLETA_LEROY/LERoyMERLIN_UNIFICADO_UPLOAD_2025-07-13.xlsx


ModuleNotFoundError: No module named 'selenium'